# Mô tả

## Mục tiêu
Trong notebook này, nhóm sẽ thực hiện:
1.  **Xây dựng giả thuyết:** Đề xuất 2 cách tiếp cận khác nhau để trả lời câu hỏi ML: **dự đoán `salary_in_usd`**.
2.  **Feature Engineering:** Biến đổi dữ liệu thô thành các features phù hợp cho từng giả thuyết.
3.  **Data Splitting:** Chia tập dữ liệu thành Train/Test và lưu trữ để chuẩn bị cho bước Modeling.

## Input & Output
- **Input:** `../data/drop_dup_data.csv` (Dữ liệu gốc đã drop duplicated).
- **Output:** - `../data/train_hypo_1.csv`, `../data/test_hypo_1.csv`
    - `../data/train_hypo_2.csv`, `../data/test_hypo_2.csv`

## Import thư viện

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split


# Cấu hình hiển thị để dễ quan sát dữ liệu
pd.set_option('display.max_columns', None)

# Đường dẫn file dữ liệu (Cập nhật nếu cần)
FILE_PATH = '../data/drop_dup_data.csv'
OUTPUT_DIR = '../data/'

# Đọc dữ liệu
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã đọc dữ liệu thành công. Kích thước: {df.shape}")
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file tại {FILE_PATH}")


Đã đọc dữ liệu thành công. Kích thước: (52437, 12)


# 1. Giả thuyết 1: Thâm niên và vị trí địa lý

## 1.1. Cơ sở lựa chọn
Dựa trên quá trình EDA và quan sát thực tế thị trường lao động, nhóm nhận thấy:
1.  **Kinh nghiệm (Experience Level):** Có tương quan dương rất mạnh với mức lương. Đường nối các điểm lương trung vị của từng mức kinh nghiệm là một đường thẳng tuyến tính đồng biến.
2.  **Vị trí công ty (Company Location):** Mỹ (US) là thị trường chi trả lương cao nhất thế giới cho ngành dữ liệu. Sự chênh lệch giữa công ty tại Mỹ và các quốc gia khác là rất lớn, lấn át các yếu tố địa lý nhỏ lẻ khác.
3.  **Lạm phát theo năm (Work Year):** Lương có xu hướng tăng nhẹ theo từng năm do lạm phát và nhu cầu thị trường.

## 1.2. Mục tiêu thực nghiệm
Mục tiêu của giả thuyết này là xây dựng một **Baseline Model**. 

Nhóm muốn kiểm chứng xem: Liệu chỉ với 3 yếu tố cốt lõi là:
- **"Bạn có nhiều kinh nghiệm thế nào?"** (Experience)
- **"Công ty bạn ở đâu?"** (US/Non-US)
- **"Thời điểm nào?"** (Year) 

Đã đủ để giải thích phần lớn sự biến thiên của mức lương hay chưa? 

Nếu mô hình này có kết quả tốt, chứng tỏ thị trường định giá nhân sự chủ yếu dựa trên thâm niên và địa lý.

## 1.3. Phương pháp thực hiện

Để đưa các dữ liệu trên vào mô hình hồi quy, nhóm thực hiện Feature Engineering như sau:

* **Experience Level:** Sử dụng **Ordinal Encoding** (Mã hóa thứ tự) vì cấp bậc có tính chất tăng dần: EN (0) < MI (1) < SE (2) < EX (3).
* **Company Location:** Sử dụng **Binary Encoding** (Mã hóa nhị phân). Gom nhóm tất cả các quốc gia thành 2 nhóm: `1` (nếu là US) và `0` (nếu không phải US). Điều này giúp giảm chiều dữ liệu đáng kể so với One-hot encoding hàng chục quốc gia.
* **Work Year:** Giữ nguyên dạng số thực để mô hình bắt được xu hướng tuyến tính theo thời gian.

## 1.4. Hàm giả thuyết

Với mô hình Linear Regression, phương trình dự kiến sẽ có dạng:

$$h_\theta(x) = \theta_0 + \theta_1 * (\text{experience\_level}) + \theta_2 * (\text{is\_US\_company}) + \theta_3 * (\text{work\_year})$$

Trong đó:
* $h_\theta(x)$: Mức lương dự đoán (`salary_in_usd`).
* $\theta_0$: Bias.
* $\theta_1, \theta_2, \theta_3$: Trọng số hồi quy cho từng đặc trưng. Nhóm kỳ vọng các hệ số này đều mang dấu dương (+).

## 1.5. Code

### Helper Functions

In [3]:
# --- Helper Functions cho H1 ---

def map_experience_level(level):
    """
    Chuyển đổi kinh nghiệm từ chuỗi ký tự sang số nguyên (Ordinal).
    Input: 'EN', 'MI', 'SE', 'EX'
    Output: 0, 1, 2, 3
    """
    mapping = {'EN': 0, 'MI': 1, 'SE': 2, 'EX': 3}
    return mapping.get(level, 0) # Mặc định EN nếu lỗi

def create_hypothesis_1_data(df_input):
    """
    Feature Engineering cho Giả thuyết 1.
    Input: DataFrame gốc
    Output: DataFrame features H1
    """
    df_temp = df_input.copy()
    
    # 1. Encode Experience
    df_temp['experience_level_encoded'] = df_temp['experience_level'].apply(map_experience_level)
    
    # 2. Encode Location (US vs Non-US)
    df_temp['is_US_company'] = df_temp['company_location'].apply(lambda x: 1 if x == 'US' else 0)
    
    # 3. Chọn features
    features = ['work_year', 'experience_level_encoded', 'is_US_company', 'salary_in_usd']
    
    return df_temp[features]

### Thực thi

In [4]:
# --- Thực thi ---
print("--- Đang xử lý Giả thuyết 1 ---")
df_h1 = create_hypothesis_1_data(df)

# Hiển thị 5 dòng đầu để kiểm tra
display(df_h1.head())
print(f"Kích thước tập dữ liệu H1: {df_h1.shape}")

--- Đang xử lý Giả thuyết 1 ---


,work_year,experience_level_encoded,is_US_company,salary_in_usd
0,2025,0,0,69120
1,2025,0,0,50160
2,2025,0,1,158113
3,2025,0,1,87795
4,2025,3,1,351410


Kích thước tập dữ liệu H1: (52437, 4)


### Chia tập train, test

In [5]:
# Tách features (X) và target (y)
X_h1 = df_h1.drop(columns=['salary_in_usd'])
y_h1 = df_h1['salary_in_usd']

# Chia tập Train/Test theo tỷ lệ 80/20
# random_state=42 giúp cố định kết quả chia ngẫu nhiên để tái lập thí nghiệm sau này
X_train_h1, X_test_h1, y_train_h1, y_test_h1 = train_test_split(X_h1, y_h1, test_size=0.2, random_state=42)

# Ghép lại thành DataFrame để lưu file tiện lợi
train_h1 = pd.concat([X_train_h1, y_train_h1], axis=1)
test_h1 = pd.concat([X_test_h1, y_test_h1], axis=1)

# Lưu ra file CSV
train_path_h1 = os.path.join(OUTPUT_DIR, 'train_hypo_1.csv')
test_path_h1 = os.path.join(OUTPUT_DIR, 'test_hypo_1.csv')

train_h1.to_csv(train_path_h1, index=False)
test_h1.to_csv(test_path_h1, index=False)

print(f"Đã lưu file Train H1 tại: {train_path_h1} | Shape: {train_h1.shape}")
print(f"Đã lưu file Test H1 tại: {test_path_h1}   | Shape: {test_h1.shape}")

Đã lưu file Train H1 tại: ../data/train_hypo_1.csv | Shape: (41949, 4)
Đã lưu file Test H1 tại: ../data/test_hypo_1.csv   | Shape: (10488, 4)


# 2. Giả thuyết 2: Chức vụ công việc (Job Role) và Quy mô công ty

## 2.1. Cơ sở lựa chọn
Giả thuyết 1 có thể quá đơn giản hóa. Thực tế, lương còn phụ thuộc vào các yếu tố chi tiết hơn:
1.  **Chức danh (Job Role):** Một "Data Scientist" thường có lương khác với "Data Analyst" hay "Data Manager", dù cùng số năm kinh nghiệm.
2.  **Quy mô công ty (Company Size):** Các công ty lớn (Large) thường có nguồn lực tài chính mạnh hơn các công ty nhỏ (Small), dẫn đến khung lương khác nhau.
3.  **Làm việc từ xa (Remote Ratio):** Hình thức làm việc (Remote/On-site) cũng là một yếu tố phản ánh văn hóa công ty và có thể ảnh hưởng đến lương (ví dụ: remote cho công ty nước ngoài).

## 2.2. Mục tiêu thực nghiệm
Mục tiêu là kiểm tra xem **"Độ phức tạp có mang lại hiệu quả?"**.

Nhóm muốn quan sát xem việc thêm các biến chi tiết (Role, Size, Remote) có giúp giảm sai số dự báo (RMSE) và tăng độ chính xác ($R^2$) đáng kể so với Baseline Model ở H1 hay không. Đồng thời, nhóm muốn xem mô hình đánh giá vai trò nào (Scientist, Engineer, hay Manager) đóng góp tích cực nhất vào mức lương.

## 2.3. Phương pháp thực hiện

* **Job Title:** Vì có quá nhiều chức danh (gây nhiễu), nhóm sẽ **Gom nhóm (Grouping)** các chức danh tương đồng (ví dụ: 'Head of Data', 'Data Lead' -> 'Manager_Lead'), sau đó sử dụng **One-Hot Encoding**.
* **Company Size:** Sử dụng **Ordinal Encoding** (S < M < L) vì quy mô có tính thứ tự.
* **Remote Ratio:** Giữ nguyên giá trị số (0, 50, 100) hoặc coi như biến định lượng.
* **Experience Level & Work Year:** Giữ nguyên cách xử lý như H1 vì đây là các biến nền tảng quan trọng.

## 2.4. Hàm giả thuyết
Với giả thuyết này, phương trình hồi quy tuyến tính mở rộng sẽ là:

$$h_\theta(x) = \theta_0 + \theta_1(Exp) + \theta_2(Size) + \theta_3(Remote) + \theta_4(WorkYear) + \sum_{i=1}^{k} \theta_{5,i} * (JobRole\_i)$$

Trong đó:
* $\theta_0$: Hệ số chặn (Bias).
* $\theta_1(Exp)$: Hệ số cho kinh nghiệm (Experience Level).
* $\theta_2(Size)$: Hệ số cho quy mô công ty (Company Size).
* $\theta_3(Remote)$: Hệ số cho tỷ lệ làm việc từ xa (Remote Ratio).
* $\theta_4(WorkYear)$: Hệ số cho năm làm việc (để phản ánh lạm phát/tăng trưởng lương theo thời gian).
* $\sum \theta_{5,i} * (JobRole\_i)$: Tổng các hệ số tương ứng với các nhóm nghề nghiệp sau khi đã One-hot encoding (ví dụ: $\theta_{Scientist} * Is\_Scientist$).

## 2.5. Code

### Helper Functions

In [6]:
# --- Helper Functions cho H2 ---

def map_company_size(size):
    """
    Chuyển đổi quy mô công ty sang giá trị số (Ordinal Encoding).
    
    Input: size (str) - 'S', 'M', 'L'
    Output: int - 0, 1, 2
    """
    mapping = {'S': 0, 'M': 1, 'L': 2}
    return mapping.get(size, 1) # Mặc định là M nếu thiếu dữ liệu

def categorize_job_title(title):
    """
    Gom nhóm các Job Title chi tiết thành các nhóm lớn (Reducing Cardinality).
    
    Input: title (str) - Tên chức danh gốc
    Output: str - Tên nhóm chức danh mới
    """
    title = str(title).lower()
    
    # Ưu tiên tìm kiếm theo từ khóa
    if any(x in title for x in ['manager', 'lead', 'principal', 'head', 'director']):
        return 'Manager_Lead'
    elif any(x in title for x in ['scientist', 'science', 'research']):
        return 'Data_Scientist'
    elif any(x in title for x in ['engineer', 'architect']):
        return 'Data_Engineer'
    elif any(x in title for x in ['analyst', 'analytics']):
        return 'Data_Analyst'
    elif any(x in title for x in ['machine learning', 'ml', 'ai', 'vision', 'nlp']):
        return 'ML_AI_Engineer'
    else:
        return 'Other'

def create_hypothesis_2_data(df_input):
    """
    Xử lý dữ liệu và trích xuất features cho Giả thuyết 2.
    """
    df_temp = df_input.copy()
    
    # 1. Ordinal Encoding: Exp & Company Size
    df_temp['experience_level_encoded'] = df_temp['experience_level'].apply(map_experience_level)
    df_temp['company_size_encoded'] = df_temp['company_size'].apply(map_company_size)
    
    # 2. Grouping Job Title
    df_temp['job_group'] = df_temp['job_title'].apply(categorize_job_title)
    
    # 3. One-Hot Encoding cho Job Group
    # drop_first=True: Loại bỏ 1 cột để tránh hiện tượng đa cộng tuyến hoàn hảo (Dummy Variable Trap)
    job_dummies = pd.get_dummies(df_temp['job_group'], prefix='job', drop_first=True, dtype=int)
    
    # Ghép các cột dummy vào dataframe tạm
    df_temp = pd.concat([df_temp, job_dummies], axis=1)
    
    # 4. Feature Selection
    # Lấy các features cơ bản + các cột dummy vừa tạo
    feature_cols = ['work_year', 'experience_level_encoded', 'company_size_encoded', 'remote_ratio']
    feature_cols.extend(job_dummies.columns.tolist()) # Thêm danh sách các cột job_...
    feature_cols.append('salary_in_usd') # Target
    
    return df_temp[feature_cols]

### Thực thi

In [7]:
# --- Thực thi ---
print("--- Đang xử lý Giả thuyết 2 ---")
df_h2 = create_hypothesis_2_data(df)

display(df_h2.head())
print(f"Kích thước tập dữ liệu H2: {df_h2.shape}")
print(f"Danh sách Features H2: {list(df_h2.columns)}")

--- Đang xử lý Giả thuyết 2 ---


,work_year,experience_level_encoded,company_size_encoded,remote_ratio,job_Data_Engineer,job_Data_Scientist,job_ML_AI_Engineer,job_Manager_Lead,job_Other,salary_in_usd
0,2025,0,1,0,0,0,0,0,1,69120
1,2025,0,1,0,0,0,0,0,1,50160
2,2025,0,1,0,1,0,0,0,0,158113
3,2025,0,1,0,1,0,0,0,0,87795
4,2025,3,1,0,1,0,0,0,0,351410


Kích thước tập dữ liệu H2: (52437, 10)
Danh sách Features H2: ['work_year', 'experience_level_encoded', 'company_size_encoded', 'remote_ratio', 'job_Data_Engineer', 'job_Data_Scientist', 'job_ML_AI_Engineer', 'job_Manager_Lead', 'job_Other', 'salary_in_usd']


### Chia tập train, test

In [8]:
# Tách features và target
X_h2 = df_h2.drop(columns=['salary_in_usd'])
y_h2 = df_h2['salary_in_usd']

# Chia tập Train/Test (Vẫn giữ random_state=42 để đồng bộ với H1 về mặt lấy mẫu)
X_train_h2, X_test_h2, y_train_h2, y_test_h2 = train_test_split(X_h2, y_h2, test_size=0.2, random_state=42)

# Ghép lại
train_h2 = pd.concat([X_train_h2, y_train_h2], axis=1)
test_h2 = pd.concat([X_test_h2, y_test_h2], axis=1)

# Lưu file CSV
train_path_h2 = os.path.join(OUTPUT_DIR, 'train_hypo_2.csv')
test_path_h2 = os.path.join(OUTPUT_DIR, 'test_hypo_2.csv')

train_h2.to_csv(train_path_h2, index=False)
test_h2.to_csv(test_path_h2, index=False)

print(f"Đã lưu file Train H2 tại: {train_path_h2} | Shape: {train_h2.shape}")
print(f"Đã lưu file Test H2 tại: {test_path_h2}   | Shape: {test_h2.shape}")

Đã lưu file Train H2 tại: ../data/train_hypo_2.csv | Shape: (41949, 10)
Đã lưu file Test H2 tại: ../data/test_hypo_2.csv   | Shape: (10488, 10)


# 3. Cải tiến thêm các giả thuyết

Sau khi thực hiện xong 2 giả thuyết trên và ra kết quả rất thấp, bị underfit siêu nặng, thầy cũng đã góp ý và đưa ra một số ý tưởng cho nhóm thử cải tiến.

Nhóm quyết định sẽ thử các bước cải tiến của thầy, tạo các feature mới, dùng log để biến đổi.

In [9]:
# === Step 1: Feature selection bằng Linear Regression + log(target) ===

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

# 1) Chọn tập feature gốc để thử (có thể thêm/bớt sau khi xem kết quả)
candidate_features = [
    "work_year",
    "experience_level",
    "employment_type",
    "job_title",
    "company_location",
    "employee_residence",
    "company_size",
    "remote_ratio",
]

target_col = "salary_in_usd"

# chỉ lấy những cột thực sự tồn tại trong df
feature_cols = [c for c in candidate_features if c in df.columns]
missing = [c for c in candidate_features if c not in df.columns]
print("Using features:", feature_cols)
if missing:
    print("Missing columns (ignored):", missing)

df_fs = df[feature_cols + [target_col]].dropna().copy()

# 2) log-transform target
y = np.log1p(df_fs[target_col].astype(float))
X = df_fs[feature_cols]

# 3) tách train/test để ranking đỡ bị “ảo”
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4) preprocess: one-hot cho categorical, passthrough numeric
numeric_cols = [c for c in feature_cols if c in ["work_year", "remote_ratio"]]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="drop",
)

# scale để coefficients so sánh được (sparse-friendly)
model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("scale", StandardScaler(with_mean=False)),
        ("linreg", LinearRegression()),
    ]
)

model.fit(X_train, y_train)

# 5) metric nhanh trên log-scale
yhat_train = model.predict(X_train)
yhat_test = model.predict(X_test)

rmse_train = np.sqrt(mean_squared_error(y_train, yhat_train))
rmse_test = np.sqrt(mean_squared_error(y_test, yhat_test))
r2_train = r2_score(y_train, yhat_train)
r2_test = r2_score(y_test, yhat_test)

print(f"Train R2 (log): {r2_train:.4f} | RMSE (log): {rmse_train:.4f}")
print(f"Test  R2 (log): {r2_test:.4f} | RMSE (log): {rmse_test:.4f}")

# 6) Lấy coefficient theo từng feature sau transform
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefs = model.named_steps["linreg"].coef_

orig_cols_all = numeric_cols + categorical_cols

def map_to_original(transformed_name: str) -> str:
    # transformed_name dạng: "num__work_year" hoặc "cat__job_title_Data Scientist"
    bare = transformed_name.split("__", 1)[-1]
    # match theo prefix cột gốc, ưu tiên cột tên dài hơn
    for col in sorted(orig_cols_all, key=len, reverse=True):
        if bare == col or bare.startswith(col + "_"):
            return col
    return bare

importance = (
    pd.DataFrame({"transformed": feature_names, "abs_coef": np.abs(coefs)})
      .assign(original=lambda d: d["transformed"].map(map_to_original))
      .groupby("original", as_index=False)["abs_coef"].sum()
      .sort_values("abs_coef", ascending=False)
)

print("\nTop 5 original features by sum(|coef|):")
display(importance.head(5))

print("\nFull ranking:")
display(importance)

Using features: ['work_year', 'experience_level', 'employment_type', 'job_title', 'company_location', 'employee_residence', 'company_size', 'remote_ratio']
Train R2 (log): 0.4170 | RMSE (log): 0.3986
Test  R2 (log): 0.4139 | RMSE (log): 0.4050

Top 5 original features by sum(|coef|):


,original,abs_coef
5,job_title,1.879263
2,employee_residence,0.725897
0,company_location,0.567254
4,experience_level,0.224846
3,employment_type,0.035970



Full ranking:


,original,abs_coef
5,job_title,1.879263
2,employee_residence,0.725897
0,company_location,0.567254
4,experience_level,0.224846
3,employment_type,0.035970
1,company_size,0.017726
6,remote_ratio,0.011278
7,work_year,0.005021


In [10]:
# === Step 2: Tạo Hypothesis H3 (top-5 features) + chia Train/Valid/Test ===

import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# Top-5 features từ Step 1
h3_features = [
    "job_title",
    "employee_residence",
    "company_location",
    "experience_level",
    "employment_type",
]
target_col = "salary_in_usd"

df_h3 = df[h3_features + [target_col]].dropna().copy()

# (Optional) thêm target log để tiện dùng ở notebook 05
df_h3["salary_log1p"] = np.log1p(df_h3[target_col].astype(float))

# Split 70/15/15
RANDOM_STATE = 42
test_size = 0.15
valid_size = 0.15
valid_ratio_of_trainval = valid_size / (1 - test_size)  # = 0.176470588...

trainval_df, test_df = train_test_split(df_h3, test_size=test_size, random_state=RANDOM_STATE)
train_df, valid_df = train_test_split(
    trainval_df, test_size=valid_ratio_of_trainval, random_state=RANDOM_STATE
)

print("H3 shapes:")
print("  train:", train_df.shape)
print("  valid:", valid_df.shape)
print("  test :", test_df.shape)

OUTPUT_DIR = "../data/"
train_path = os.path.join(OUTPUT_DIR, "train_hypo_3.csv")
valid_path = os.path.join(OUTPUT_DIR, "valid_hypo_3.csv")
test_path  = os.path.join(OUTPUT_DIR, "test_hypo_3.csv")

train_df.to_csv(train_path, index=False)
valid_df.to_csv(valid_path, index=False)
test_df.to_csv(test_path, index=False)

print("Saved:")
print(" ", train_path)
print(" ", valid_path)
print(" ", test_path)
print("H3 columns:", list(train_df.columns))

H3 shapes:
  train: (36705, 7)
  valid: (7866, 7)
  test : (7866, 7)
Saved:
  ../data/train_hypo_3.csv
  ../data/valid_hypo_3.csv
  ../data/test_hypo_3.csv
H3 columns: ['job_title', 'employee_residence', 'company_location', 'experience_level', 'employment_type', 'salary_in_usd', 'salary_log1p']
